# <font color="#418FDE" size="6.5" uppercase>**Klassische Bildmodelle**</font>

>Last update: 20260723.
    
By the end of this Lecture, you will be able to:
- Bereiten kleine Bilddaten als Merkmalsvektoren oder klassische Bildmerkmale vor. 
- Trainieren klassische Klassifikatoren auf Bildmerkmalen ohne Datenleckage. 
- Bewerten Bildmodelle mit Konfusionsmatrix und visualisierten Fehlklassifikationen. 


## **1. Bildmerkmale vorbereiten**

### **1.1. Digits Datensatz laden**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_13/Lecture_A/image_01_01.jpg?v=1784804031" width="250">



>* Kleine Graustufenbilder handgeschriebener Ziffern
>* Praxisnah, überschaubar und vielfältig für Bildklassifikation

>* Graustufenbilder als geordnete Pixelstrukturen verstehen
>* Merkmale und Zielklassen klar trennen

>* Grunddaten und Klassenverteilung früh prüfen
>* Beispielbilder auf mögliche Verwechslungen untersuchen



In [ ]:
#@title Python-Code - Digits Datensatz laden

# Wir laden kleine handgeschriebene Ziffernbilder.
# Bildform und Zielklassen werden getrennt betrachtet.
# Eine Beispielziffer wird als Graustufenbild angezeigt.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

# Der Datensatz ist lokal in scikit-learn enthalten.
digits = load_digits()
images = digits.images
targets = digits.target

# Diese Prüfung schützt vor unerwarteten Datenformen.
if images.ndim != 3 or targets.ndim != 1:
    raise ValueError("Die Bilddaten oder Zielwerte haben eine unerwartete Form.")

# Jedes Bild kann später zu einem Merkmalsvektor werden.
feature_vectors = images.reshape(images.shape[0], -1)

# Kurze Kennzahlen zeigen die Organisation der Daten.
print(f"Anzahl Bilder: {images.shape[0]}")
print(f"Bildgröße: {images.shape[1]} x {images.shape[2]} Pixel")
print(f"Merkmale pro Bild nach Umformung: {feature_vectors.shape[1]}")
print(f"Helligkeitswerte: {images.min():.0f} bis {images.max():.0f}")
print(f"Zielklassen: {np.unique(targets).tolist()}")

# Ein einzelnes Bild macht die Pixelstruktur sichtbar.
example_index = 0
example_image = images[example_index]
example_label = targets[example_index]

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(example_image, cmap="gray_r", vmin=0, vmax=16)
ax.set_title(f"Beispielbild mit Zielklasse {example_label}")
ax.set_xlabel("Pixelspalte")
ax.set_ylabel("Pixelzeile")
plt.show()



### **1.2. Pixel als Vektoren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_13/Lecture_A/image_01_02.jpg?v=1784804032" width="250">



>* Bildraster werden zu Zahlenfolgen umgeformt
>* Jeder Vektoreintrag beschreibt eine Pixelhelligkeit

>* Gleiche Pixelreihenfolge sichert feste Bedeutung.
>* Modelle erkennen ortsgebundene Helligkeitsmuster.

>* Verschiebungen verändern Pixelvektoren stark.
>* Vorverarbeitung reduziert Aufwand und Fehlmuster.



In [ ]:
#@title Python-Code - Pixel als Vektoren

# Wir verwandeln ein kleines Bild in Zahlen.
# Pixelwerte werden zeilenweise zu einem Vektor.
# Die Ausgabe zeigt Form, Reihenfolge und Bedeutung.

import numpy as np
import matplotlib.pyplot as plt

# Dieses synthetische Graustufenbild ist klein und übersichtlich.
image = np.array(
    [[0, 0, 255, 0], [0, 255, 255, 0], [0, 0, 255, 0], [0, 255, 255, 255]],
    dtype=np.uint8,
)

# Die Form beschreibt Höhe und Breite des Bildrasters.
height, width = image.shape
pixel_count = height * width

# Diese Prüfung macht die erwartete Bildstruktur ausdrücklich sichtbar.
if image.ndim != 2:
    raise ValueError("Erwartet wird ein zweidimensionales Graustufenbild.")

# reshape legt die Pixel zeilenweise hintereinander.
vector = image.reshape(-1)

# Die ersten Werte zeigen die feste Reihenfolge im Vektor.
first_values = vector[:12].tolist()
print(f"Bildform: {height} x {width} Pixel")
print(f"Vektorlänge: {len(vector)} Merkmale")
print(f"Erste 12 Vektorwerte: {first_values}")

# Ein einzelner Vektoreintrag gehört zu genau einem Bildort.
row = 1
column = 2
index = row * width + column
print(f"Pixel an Zeile {row}, Spalte {column}: {image[row, column]}")
print(f"Derselbe Wert im Vektor an Position {index}: {vector[index]}")

# Die Grafik zeigt das Bild, dessen Pixel vektorisiert wurden.
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(image, cmap="gray", vmin=0, vmax=255)
ax.set_title("Synthetisches 4x4-Graustufenbild")
ax.set_xlabel("Spalte")
ax.set_ylabel("Zeile")
plt.show()



### **1.3. Histogramme als Merkmale**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_13/Lecture_A/image_01_03.jpg?v=1784804034" width="250">



>* Histogramme zählen Helligkeitsbereiche statt Pixelpositionen
>* Robuster gegen Verschiebung, Rauschen und Verformung

>* Bin-Größe bestimmt Detailgrad und Rauschempfindlichkeit
>* Histogramme fassen relevante Bildverteilungen kompakt zusammen

>* Histogramme einheitlich berechnen und normalisieren
>* Räumliche Information durch Zusatzmerkmale ergänzen



In [ ]:
#@title Python-Code - Histogramme als Merkmale

# Dieses Beispiel zeigt Histogramme als Bildmerkmale.
# Helligkeitsbereiche ersetzen einzelne Pixelpositionen.
# Das Diagramm vergleicht zwei einfache Ziffernbilder.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

# Wir laden kleine Grauwertbilder aus scikit-learn.
digits = load_digits()
images = digits.images
labels = digits.target

# Eine einfache Prüfung macht die Bildform verständlich.
if images.ndim != 3 or images.shape[1:] != (8, 8):
    raise ValueError("Erwartet werden kleine 8-mal-8-Grauwertbilder.")

# Wir wählen je ein Beispiel für zwei Ziffern.
first_zero_index = np.where(labels == 0)[0][0]
first_eight_index = np.where(labels == 8)[0][0]
zero_image = images[first_zero_index]
eight_image = images[first_eight_index]

# Dieselben Wertebereiche erzeugen vergleichbare Merkmalsvektoren.
bin_edges = np.array([0, 4, 8, 12, 16, 17])
zero_histogram, _ = np.histogram(zero_image, bins=bin_edges)
eight_histogram, _ = np.histogram(eight_image, bins=bin_edges)

# Normalisierung macht die Werte unabhängig von der Pixelanzahl.
zero_features = zero_histogram / zero_histogram.sum()
eight_features = eight_histogram / eight_histogram.sum()

# Kurze Ausgabe zeigt die fertigen Histogrammmerkmale.
print("Datensatz: sklearn load_digits mit 8x8-Grauwertbildern")
print("Histogramm-Merkmale für Ziffer 0:", np.round(zero_features, 2))
print("Histogramm-Merkmale für Ziffer 8:", np.round(eight_features, 2))

# Ein Balkendiagramm vergleicht die Helligkeitsverteilungen.
bin_labels = ["0-3", "4-7", "8-11", "12-15", "16"]
x_positions = np.arange(len(bin_labels))
bar_width = 0.35

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x_positions - bar_width / 2, zero_features, bar_width, label="Ziffer 0")
ax.bar(x_positions + bar_width / 2, eight_features, bar_width, label="Ziffer 8")

ax.set_title("Normalisierte Helligkeitshistogramme als Merkmale")
ax.set_xlabel("Grauwertbereich")
ax.set_ylabel("Anteil der Pixel")
ax.set_xticks(x_positions)

ax.set_xticklabels(bin_labels)
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()

plt.show()



## **2. Kanten und HOG**

### **2.1. Kanten als Merkmale**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_13/Lecture_A/image_02_01.jpg?v=1784804019" width="250">



>* Kanten zeigen wichtige Helligkeits- und Farbwechsel
>* Sie betonen Formen statt empfindlicher Pixelwerte

>* Gradienten erzeugen Kantenkarten aus lokalen Änderungen
>* Konsistente Vorverarbeitung macht Defekte vergleichbar

>* Merkmalsextraktion nur mit Trainingsdaten anpassen
>* Saubere Trennung verhindert Datenleckage



In [ ]:
#@title Python-Code - Kanten als Merkmale

# Wir trainieren ein kleines Bildmodell mit Kantenmerkmalen.
# Die Skalierung wird nur auf Trainingsdaten gelernt.
# Die Ausgabe vergleicht Pixel und Kanten fair.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Der Digits-Datensatz enthält kleine Grauwertbilder.
digits = load_digits()
images = digits.images.astype(np.float32)
labels = digits.target

# Eine einfache Prüfung macht die Bildform explizit.
if images.shape[1:] != (8, 8):
    raise ValueError("Erwartet werden 8 mal 8 Pixel pro Bild.")

# Die Aufteilung passiert vor jeder gelernten Vorverarbeitung.
train_images, test_images, y_train, y_test = train_test_split(
    images, labels, test_size=0.25, stratify=labels, random_state=42
)

# Rohe Pixel werden nur flach als Vergleichsmerkmale genutzt.
X_train_pixels = train_images.reshape(len(train_images), -1)
X_test_pixels = test_images.reshape(len(test_images), -1)

# Gradienten messen Helligkeitsänderungen in zwei Richtungen.
train_grad_y, train_grad_x = np.gradient(train_images, axis=(1, 2))
test_grad_y, test_grad_x = np.gradient(test_images, axis=(1, 2))

# Die Kantenstärke fasst beide Richtungen pro Pixel zusammen.
train_edges = np.sqrt(train_grad_x ** 2 + train_grad_y ** 2)
test_edges = np.sqrt(test_grad_x ** 2 + test_grad_y ** 2)

# Auch die Kantenkarten werden zu Merkmalsvektoren abgeflacht.
X_train_edges = train_edges.reshape(len(train_edges), -1)
X_test_edges = test_edges.reshape(len(test_edges), -1)

# Die Pipeline verhindert Datenleckage beim Skalieren.
pixel_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42),
)

# Das zweite Modell nutzt dieselbe saubere Trainingslogik.
edge_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42),
)

# Beide Modelle werden ausschließlich mit Trainingsdaten angepasst.
pixel_model.fit(X_train_pixels, y_train)
edge_model.fit(X_train_edges, y_train)

# Die Testdaten bleiben bis zur Bewertung ungesehen.
pixel_accuracy = accuracy_score(y_test, pixel_model.predict(X_test_pixels))
edge_accuracy = accuracy_score(y_test, edge_model.predict(X_test_edges))

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Testgenauigkeit mit rohen Pixeln: {pixel_accuracy:.3f}")
print(f"Testgenauigkeit mit Kantenmerkmalen: {edge_accuracy:.3f}")

# Ein Beispielbild zeigt, was das Kantenmodell sieht.
example_index = 0
example_edges = test_edges[example_index]

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(example_edges, cmap="gray")
ax.set_title("Kantenkarte eines Testbildes")
ax.set_xlabel("Pixelspalte")
ax.set_ylabel("Pixelzeile")
plt.show()



### **2.2. HOG Merkmale**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_13/Lecture_A/image_02_02.jpg?v=1784804023" width="250">



>* HOG beschreibt Bilder über lokale Kantenrichtungen
>* Formmuster werden zu nützlichen Merkmalsvektoren

>* Zellen zählen gewichtete Kantenrichtungen.
>* Normalisierung macht HOG robust und kompakt.

>* Daten zuerst sauber aufteilen
>* Lernende Schritte nur am Training anpassen



In [ ]:
#@title Python-Code - HOG Merkmale

# Wir trainieren ein kleines HOG-Bildmodell.
# HOG beschreibt lokale Kantenrichtungen als Merkmale.
# Die Auswertung zeigt Genauigkeit ohne Datenleckage.

import numpy as np
import matplotlib.pyplot as plt
from sklearn import __version__ as sklearn_version
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score

# Wir laden kleine Ziffernbilder aus scikit-learn.
digits = load_digits()
images = digits.images.astype(np.float32)
labels = digits.target

# Diese Prüfung macht die erwartete Bildform explizit.
if images.ndim != 3 or images.shape[1:] != (8, 8):
    raise ValueError("Erwartet werden kleine Graustufenbilder mit 8 mal 8 Pixeln.")

# Wir trennen zuerst, damit spätere Anpassungen sauber bleiben.
train_images, test_images, y_train, y_test = train_test_split(
    images, labels, test_size=0.25, random_state=42, stratify=labels
)

# Diese Funktion berechnet einfache HOG-ähnliche Merkmale.
def extract_hog_features(image_batch):
    feature_rows = []
    for image in image_batch:
        gy, gx = np.gradient(image)
        magnitude = np.sqrt(gx * gx + gy * gy)
        angle = (np.degrees(np.arctan2(gy, gx)) + 180.0) % 180.0
        cell_features = []
        for row in range(0, 8, 4):
            for col in range(0, 8, 4):
                cell_mag = magnitude[row:row + 4, col:col + 4]
                cell_ang = angle[row:row + 4, col:col + 4]
                hist, _ = np.histogram(
                    cell_ang, bins=9, range=(0.0, 180.0), weights=cell_mag
                )
                cell_features.extend(hist)
        feature_rows.append(cell_features)
    return np.array(feature_rows, dtype=np.float32)

# HOG wird getrennt für Trainings- und Testbilder berechnet.
X_train = extract_hog_features(train_images)
X_test = extract_hog_features(test_images)

# Der Skalierer wird nur auf Trainingsmerkmalen angepasst.
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)

# Das Modell lernt ausschließlich aus den Trainingsdaten.
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

# Kurze Ausgaben verbinden Merkmalsform und Bewertung.
print(f"scikit-learn-Version: {sklearn_version}")
print(f"HOG-Merkmale pro Bild: {X_train.shape[1]}")
print(f"Testgenauigkeit ohne Datenleckage: {accuracy:.3f}")

# Die Konfusionsmatrix zeigt typische Verwechslungen der Ziffern.
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, colorbar=False)
ax.set_title("Konfusionsmatrix für HOG-Merkmale")
ax.set_xlabel("Vorhergesagte Ziffer")
ax.set_ylabel("Wahre Ziffer")
plt.tight_layout()
plt.show()



### **2.3. Leakage freier Split**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_13/Lecture_A/image_02_03.jpg?v=1784804021" width="250">



>* Nur Trainingsdaten bestimmen Vorverarbeitung und Merkmale
>* Saubere Splits verhindern zu optimistische Bewertungen

>* Zusammengehörige Bilder gemeinsam splitten
>* So lernt das Modell echte Klassen

>* Unabhängige Daten sauber und stratifiziert aufteilen
>* Nur Training anpassen, Test zuletzt bewerten



In [ ]:
#@title Python-Code - Leakage freier Split

# Wir demonstrieren einen leakagefreien Split mit Bildmerkmalen.
# Skalierung wird nur auf Trainingsdaten gelernt.
# Die Ausgabe vergleicht saubere und undichte Bewertung.

import numpy as np
import sklearn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Wir laden kleine Ziffernbilder aus scikit-learn.
digits = load_digits()
images = digits.images
target = digits.target

# Diese Prüfung macht die Bildannahme sichtbar.
if images.ndim != 3 or images.shape[1:] != (8, 8):
    raise ValueError("Erwartet werden kleine 8x8-Bilder.")

# Jedes Bild wird zu einem einfachen Merkmalsvektor.
features = images.reshape(images.shape[0], -1)
labels = target

# Der Split passiert vor jeder gelernten Vorverarbeitung.
X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.3, stratify=labels, random_state=42
)

# Sauber: Der Scaler sieht nur Trainingsdaten.
clean_scaler = StandardScaler()
X_train_clean = clean_scaler.fit_transform(X_train)
X_test_clean = clean_scaler.transform(X_test)

# Undicht: Der Scaler lernt aus allen Daten.
leaky_scaler = StandardScaler()
X_all_leaky = leaky_scaler.fit_transform(features)
X_train_leaky = X_all_leaky[: len(X_train)]
X_test_leaky = X_all_leaky[len(X_train) : len(X_train) + len(X_test)]

# Für den fairen Vergleich nutzen wir dasselbe Modell.
clean_model = KNeighborsClassifier(n_neighbors=3)
clean_model.fit(X_train_clean, y_train)
clean_predictions = clean_model.predict(X_test_clean)

# Das undichte Beispiel mischt Split und Vorverarbeitung falsch.
leaky_model = KNeighborsClassifier(n_neighbors=3)
leaky_model.fit(X_train_leaky, y_train)
leaky_predictions = leaky_model.predict(X_test_leaky)

# Die Kennzahlen zeigen, warum die Reihenfolge wichtig ist.
clean_accuracy = accuracy_score(y_test, clean_predictions)
leaky_accuracy = accuracy_score(y_test, leaky_predictions)

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Trainingsbilder: {len(X_train)}, Testbilder: {len(X_test)}")
print(f"Saubere Genauigkeit: {clean_accuracy:.3f}")
print(f"Undichte Genauigkeit: {leaky_accuracy:.3f}")
print("Merksatz: Split zuerst, dann Scaler nur auf Training fitten.")



## **3. Bildmodelle bewerten**

### **3.1. Klassifikatoren trainieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_13/Lecture_A/image_03_01.jpg?v=1784804025" width="250">



>* Bilder als Merkmalsvektoren für Klassifikatoren nutzen
>* Training und Test strikt getrennt halten

>* Leistung pro Klasse statt nur Gesamtgenauigkeit
>* Sauberes Training macht Fehler sichtbar

>* Fehlerbilder zeigen Ursachen falscher Vorhersagen
>* Beobachtungen leiten gezielte Modellverbesserungen ab



In [ ]:
#@title Python-Code - Klassifikatoren trainieren

# Wir trainieren einen einfachen Bildklassifikator.
# Saubere Pipelines verhindern Datenleckage beim Skalieren.
# Die Konfusionsmatrix zeigt typische Verwechslungen.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import ConfusionMatrixDisplay

# Der Datensatz enthält kleine Grauwertbilder handgeschriebener Ziffern.
digits = load_digits()
images = digits.images
target = digits.target

# Diese Prüfung macht die erwartete Bildform sichtbar.
if images.shape[1:] != (8, 8):
    raise ValueError("Erwartet werden 8 mal 8 Pixel pro Bild.")

# Jedes Bild wird zu einem Merkmalsvektor abgeflacht.
features = images.reshape(images.shape[0], -1)

# Die Aufteilung bleibt stratifiziert, damit alle Ziffern vertreten sind.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.25,
    random_state=42,
    stratify=target,
)

# Die Skalierung wird nur auf Trainingsdaten gelernt.
model = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

# Der Klassifikator lernt ausschließlich aus den Trainingsdaten.
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Kurze Kennzahlen bereiten die visuelle Bewertung vor.
accuracy = accuracy_score(y_test, y_pred)
print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Testgenauigkeit: {accuracy:.3f}")
print(f"Trainingsbilder: {len(X_train)}, Testbilder: {len(X_test)}")

# Eine Konfusionsmatrix zeigt, welche Ziffern verwechselt werden.
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    ax=ax,
    cmap="Blues",
    colorbar=False,
)

ax.set_title("Konfusionsmatrix für Ziffernbilder")
ax.set_xlabel("Vorhergesagte Ziffer")
ax.set_ylabel("Wahre Ziffer")
plt.tight_layout()
plt.show()



### **3.2. PCA Ergebnisse prüfen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_13/Lecture_A/image_03_02.jpg?v=1784804027" width="250">



>* PCA-Darstellung auf sinnvolle Bildinformation prüfen
>* Zu starke Reduktion kann Klassenhinweise verlieren

>* PCA-Projektionen zeigen Klassentrennung im Merkmalsraum
>* Fehlklassifikationen mit Datenstruktur gemeinsam deuten

>* Hauptkomponenten visuell und kritisch deuten
>* Fehlerbilder mit PCA und Konfusionsmatrix verbinden



In [ ]:
#@title Python-Code - PCA Ergebnisse prüfen

# Wir prüfen PCA-Ergebnisse an kleinen Ziffernbildern.
# Die Projektion zeigt Nähe und mögliche Verwechslungen.
# Die Grafik verbindet Klassenstruktur mit Modellfehlern.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Die Ziffernbilder sind klein und bereits lokal verfügbar.
digits = load_digits()
X = digits.data
y = digits.target

# Eine einfache Prüfung schützt vor unerwarteten Datenformen.
if X.shape[0] != y.shape[0] or X.shape[1] != 64:
    raise ValueError("Die Zifferndaten haben nicht die erwartete Form.")

# Der Split verhindert Datenleckage vor PCA und Klassifikation.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# PCA wird nur auf Trainingsdaten im Pipeline-Ablauf gelernt.
model = make_pipeline(
    StandardScaler(),
    PCA(n_components=2, random_state=42),
    LogisticRegression(max_iter=1000, random_state=42)
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

# Für die Grafik betrachten wir die Testdaten im PCA-Raum.
scaler = model.named_steps["standardscaler"]
pca = model.named_steps["pca"]
X_test_pca = pca.transform(scaler.transform(X_test))

# Falsch klassifizierte Punkte markieren mögliche Problemzonen.
wrong_mask = y_pred != y_test
explained = pca.explained_variance_ratio_.sum()

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Testgenauigkeit mit zwei PCA-Komponenten: {accuracy:.3f}")
print(f"Erklärte Varianz der zwei Komponenten: {explained:.3f}")

# Eine einzelne PCA-Projektion macht Überlappungen sichtbar.
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(
    X_test_pca[:, 0], X_test_pca[:, 1], c=y_test, cmap="tab10", s=28, alpha=0.75
)

ax.scatter(
    X_test_pca[wrong_mask, 0], X_test_pca[wrong_mask, 1],
    facecolors="none", edgecolors="black", s=90, label="falsch klassifiziert"
)

ax.set_title("PCA-Projektion der Testbilder mit Fehlklassifikationen")
ax.set_xlabel("Erste Hauptkomponente")
ax.set_ylabel("Zweite Hauptkomponente")
ax.legend(loc="best")

plt.show()



### **3.3. Eigenes Bildprojekt**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_13/Lecture_A/image_03_03.jpg?v=1784804029" width="250">



>* Fehler nach Anwendungskontext und Klassen bewerten
>* Konfusionsmatrix zeigt wichtige Verwechslungen

>* Fehlerbilder zeigen Ursachen hinter Verwechslungen
>* Muster weisen auf sinnvolle Verbesserungen hin

>* Testdaten sauber trennen, passende Metriken wählen
>* Grenzen benennen und nächste Verbesserungen planen



In [ ]:
#@title Python-Code - Eigenes Bildprojekt

# Dieses Beispiel bewertet ein kleines Bildmodell.
# Die Konfusionsmatrix zeigt typische Verwechslungen.
# Fehlklassifikationen werden als konkrete Ziffernbilder sichtbar.

import numpy as np
import matplotlib.pyplot as plt
from sklearn import __version__ as sklearn_version
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Wir nutzen kleine eingebaute Ziffernbilder ohne Download.
digits = load_digits()
images = digits.images
target = digits.target

# Eine einfache Prüfung verhindert unklare Datenformen.
if images.shape[0] != target.shape[0]:
    raise ValueError("Bildanzahl und Zielwerte passen nicht zusammen.")

# Jedes Bild wird zu einem Merkmalsvektor abgeflacht.
features = images.reshape(images.shape[0], -1)
class_names = [str(label) for label in digits.target_names]

# Die Testdaten bleiben bis zur Bewertung ungesehen.
X_train, X_test, y_train, y_test, img_train, img_test = train_test_split(
    features, target, images, test_size=0.25, stratify=target, random_state=42
)

# Skalierung und Modell werden nur mit Trainingsdaten angepasst.
model = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)
)
model.fit(X_train, y_train)

# Jetzt bewerten wir ausschließlich auf den Testdaten.
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
wrong_indices = np.flatnonzero(y_pred != y_test)

print(f"scikit-learn-Version: {sklearn_version}")
print(f"Testgenauigkeit: {accuracy:.3f}")
print(f"Fehlklassifikationen im Testset: {len(wrong_indices)}")

# Die Matrix zeigt, welche Ziffern verwechselt werden.
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=class_names, cmap="Blues", ax=ax
)

ax.set_title("Konfusionsmatrix für ein eigenes Ziffern-Bildprojekt")
ax.set_xlabel("Vorhergesagte Klasse")
ax.set_ylabel("Wahre Klasse")
plt.tight_layout()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Klassische Bildmodelle**</font>


In this lecture, you learned to:
- Bereiten kleine Bilddaten als Merkmalsvektoren oder klassische Bildmerkmale vor. 
- Trainieren klassische Klassifikatoren auf Bildmerkmalen ohne Datenleckage. 
- Bewerten Bildmodelle mit Konfusionsmatrix und visualisierten Fehlklassifikationen. 

In the next Lecture (Lecture B), we will go over 'Signalmodelle'